# Custom-word lip-reading training and evaluation

This notebook contains only the active eight-word workflow for **hello, hi, yes, no, good, bad, thanks,** and **bye**. The former GRID/Kaggle experiments are preserved in `archive/kaggle_lip_reading/lipnet_kaggle.ipynb`.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

import numpy as np
import torch
import torch.nn as nn

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'data' / 'prepare_custom_word_dataset.py').is_file():
            return candidate
    raise FileNotFoundError('Could not find the VisionSpeak project root.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f'Project root: {PROJECT_ROOT}')
print(f'Device: {device}')

## Isolated-word classifier

Load the recorded custom dataset, train the eight-class 3D-CNN + BiGRU classifier, and inspect its per-word results.

In [ ]:
from data.prepare_custom_word_dataset import get_custom_words_dataloaders
from models.LBY.model import LipReadingClassifier

train_loader, test_loader, label_to_index, index_to_label = get_custom_words_dataloaders(
    test_size=0.2, seed=42, batch_size=32,
    data_dir=PROJECT_ROOT / 'data' / 'custom_words',
)
num_classes = len(label_to_index)

print(f'Device: {device}')
print(f'Train samples: {len(train_loader.dataset)}')
print(f'Test samples: {len(test_loader.dataset)}')
print(f'Classes: {num_classes} {label_to_index}')


In [ ]:
from pathlib import Path

import torch
import torch.nn as nn

model = LipReadingClassifier(num_classes=num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[40, 60], gamma=0.1)
criterion = nn.CrossEntropyLoss()

In [ ]:


out_dir = PROJECT_ROOT / 'checkpoints'
out_dir.mkdir(parents=True, exist_ok=True)

def run_epoch(loader, training):
    model.train(training)
    total_loss = correct = total = 0
    for frames, labels in loader:
        frames, labels = frames.to(device), labels.to(device)
        with torch.set_grad_enabled(training):
            logits = model(frames)
            loss = criterion(logits, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)

epochs = 100
best_accuracy = 0.0
for epoch in range(1, epochs + 1):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    test_loss, test_acc = run_epoch(test_loader, training=False)
    scheduler.step()
    print(f'Epoch {epoch:2d}/{epochs} | train_loss {train_loss:.4f} train_acc {train_acc:.4f} | test_loss {test_loss:.4f} test_acc {test_acc:.4f}')
    if test_acc > best_accuracy:
        best_accuracy = test_acc
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'label_to_index': label_to_index, 'test_accuracy': test_acc}, out_dir / 'custom_words.pt')

print(f'Best test accuracy: {best_accuracy:.4f}')


In [ ]:
import numpy as np

checkpoint = torch.load(PROJECT_ROOT / 'checkpoints' / 'custom_words.pt', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for frames, labels in test_loader:
        frames, labels = frames.to(device), labels.to(device)
        all_preds.append(model(frames).argmax(dim=1).cpu())
        all_targets.append(labels.cpu())
    
preds = torch.cat(all_preds).numpy()
targets = torch.cat(all_targets).numpy()


print(f'Overall test accuracy: {float((preds == targets).mean()):.4f}')
print('\nPer-word accuracy:')
for word, idx in label_to_index.items():
    mask = targets == idx
    if mask.sum():
        print(f'  {word:>8}: {float((preds[mask] == idx).mean()):.4f} ({int(mask.sum())} samples)')

print('\nConfusion matrix (rows=truth, cols=prediction):')
conf = np.zeros((num_classes, num_classes), dtype=int)
for t, p in zip(targets, preds):
    conf[t, p] += 1
print('        ' + ' '.join(f'{index_to_label[i][:4]:>4}' for i in range(num_classes)))
for i in range(num_classes):
    print(f'{index_to_label[i]:>8} ' + ' '.join(f'{v:4d}' for v in conf[i]))


In [ ]:
# Convert integer targets/preds back to their original labels
index_to_label = {idx: label for label, idx in label_to_index.items()}

targets_labels = [index_to_label[int(t)] for t in targets]
preds_labels = [index_to_label[int(p)] for p in preds]

print("Target labels:", targets_labels)
print("Predicted labels:", preds_labels)

# Optional: print the first few
for t, p in zip(targets_labels[:10], preds_labels[:10]):
    print(f"True={t:<8} | Pred={p}")

## Character-level CTC comparison

Evaluate the custom-word CTC checkpoint against the same custom dataset. This section contains no GRID/Kaggle data loading.

In [ ]:
import numpy as np
from models.valente.model import LipNet, ctc_greedy_decode, tokens_to_text

checkpoint = torch.load(PROJECT_ROOT / 'checkpoints' / 'valente_custom_words_lipnet_ctc_0.852_acc.pt', map_location=device)
char_to_index = checkpoint['char_to_index']
index_to_char = {index: char for char, index in char_to_index.items()}

valente_model = LipNet(num_classes=len(char_to_index) + 1, input_channels=1).to(device)
valente_model.load_state_dict(checkpoint['model_state_dict'])



In [ ]:
valente_model.eval()
# Build true labels from the custom-word mapping
index_to_label = {idx: label for label, idx in label_to_index.items()}

all_true_labels = []
all_pred_labels = []

with torch.no_grad():
    for frames, labels in test_loader:
        frames = frames.to(device)
        lengths = valente_model.infer_input_lengths(frames)

        log_probs, output_lengths = valente_model.ctc_log_probs(frames, lengths)
        decoded_batch = ctc_greedy_decode(log_probs.cpu(), output_lengths.cpu(), 0)

        pred_texts = [tokens_to_text(tokens, index_to_char) for tokens in decoded_batch]
        true_texts = [index_to_label[int(lbl)] for lbl in labels.cpu().tolist()]

        all_true_labels.extend(true_texts)
        all_pred_labels.extend(pred_texts)

# Normalize spaces so labels match cleanly
all_true_labels = [str(x).strip() for x in all_true_labels]
all_pred_labels = [str(x).strip() for x in all_pred_labels]

# Build class list
classes = sorted(set(all_true_labels) | set(all_pred_labels))
label_to_id = {label: idx for idx, label in enumerate(classes)}

y_true = np.array([label_to_id[label] for label in all_true_labels])
y_pred = np.array([label_to_id[label] for label in all_pred_labels])

cm = np.zeros((len(classes), len(classes)), dtype=int)
for t, p in zip(y_true, y_pred):
    cm[t, p] += 1

print("Classes:", classes)
print("Confusion matrix:")
print(cm)

# Pretty print
print("      " + " ".join(f"{c:>8}" for c in classes))
for i, c in enumerate(classes):
    row = " ".join(f"{v:>8}" for v in cm[i])
    print(f"{c:>6} {row}")